# Heart Disease Detection — Distance-Based Model: K-Nearest Neighbors (CRISP-DM Stage 2)

This notebook implements a **Distance-Based model using K-Nearest Neighbors (KNN)** as a candidate champion model for heart disease prediction.

## Stage 2 Objectives
- Build a scikit-learn **Pipeline** (StandardScaler + KNeighborsClassifier) to prevent data leakage
- **Critical**: Feature scaling is essential for KNN since it relies on distance calculations
- Tune hyperparameters with **RandomizedSearchCV** (50 iterations, 5-fold CV)
- Output **probability scores** (0.0–1.0) and apply a custom **decision threshold**
- Evaluate with Classification Report, Confusion Matrix, ROC-AUC
- Perform **failure analysis** (top 10 most confident failures)
- Use **Permutation Importance** to assess feature contributions (Lab 05 methodology)
- Produce comparison metrics for group integration

## Computational Note
KNN is a **lazy learner** — it stores all training data and computes distances at prediction time. With ~350K training samples and 76 features, expect:
- Hyperparameter tuning: 15-45 minutes depending on hardware
- Permutation importance: 5-15 minutes

Consider using a subset of data for initial experimentation if needed.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    f1_score,
    accuracy_score,
    precision_recall_curve,
    auc,
    precision_score,
    recall_score,
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully!")
print(f"Random State: {RANDOM_STATE}")

Libraries imported successfully!
Random State: 42


## 2. Load Data from Data Preparation Stage

In [2]:
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")
y_train = pd.read_csv("y_train.csv").values.ravel()
y_test = pd.read_csv("y_test.csv").values.ravel()

print(f"X_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test  shape: {y_test.shape}")

X_train shape: (352827, 76)
X_test  shape: (88414, 76)
y_train shape: (352827,)
y_test  shape: (88414,)


## 3. Encode Target Variable & Check Class Imbalance

In [3]:
# Encode target: "Yes" -> 1, "No" -> 0
le = LabelEncoder()
le.fit(["No", "Yes"])
y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)

print(f"Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(f"\nTraining set target distribution:")
unique, counts = np.unique(y_train_enc, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  {le.inverse_transform([label])[0]} ({label}): {count} ({count/len(y_train_enc)*100:.2f}%)")

# Calculate class imbalance ratio
neg_count = np.sum(y_train_enc == 0)
pos_count = np.sum(y_train_enc == 1)
imbalance_ratio = neg_count / pos_count
print(f"\nClass imbalance ratio (No:Yes): {imbalance_ratio:.2f}:1")
print(f"\nNote: KNN does not have built-in class weights. We will handle imbalance via:")
print(f"  1. Using weights='distance' to give closer neighbors more influence")
print(f"  2. Adjusting the decision threshold post-prediction")

Label mapping: {np.str_('No'): np.int64(0), np.str_('Yes'): np.int64(1)}

Training set target distribution:
  No (0): 332781 (94.32%)
  Yes (1): 20046 (5.68%)

Class imbalance ratio (No:Yes): 16.60:1

Note: KNN does not have built-in class weights. We will handle imbalance via:
  1. Using weights='distance' to give closer neighbors more influence
  2. Adjusting the decision threshold post-prediction


## 4. Build Scikit-learn Pipeline

The pipeline bundles a `StandardScaler` and the `KNeighborsClassifier` together to prevent data leakage — scaling statistics are computed only from training folds during cross-validation.

**Why scaling is CRITICAL for KNN:**
- KNN uses distance metrics (Euclidean, Manhattan) to find nearest neighbors
- Features with larger scales dominate the distance calculation
- Without scaling, a feature ranging 0-1000 would overwhelm a feature ranging 0-1
- StandardScaler ensures all features contribute equally to distance calculations

In [4]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),  # CRITICAL: Normalizes features for distance calculations
    ("knn", KNeighborsClassifier(
        n_jobs=-1,  # Use all CPU cores for faster computation
    )),
])

print("Pipeline created:")
print(pipeline)

Pipeline created:
Pipeline(steps=[('scaler', StandardScaler()),
                ('knn', KNeighborsClassifier(n_jobs=-1))])


## 5. Hyperparameter Tuning with RandomizedSearchCV

We tune the following hyperparameters over **50 iterations** with **5-fold stratified CV**:

| Parameter | Description | Search Space |
|-----------|-------------|---------------|
| `n_neighbors` | Number of neighbors to consider | 3, 5, 7, 9, 11, 15, 21 |
| `weights` | Weight function for predictions | 'uniform', 'distance' |
| `metric` | Distance metric | 'euclidean', 'manhattan', 'minkowski' |

In [ ]:
param_distributions = {
    "knn__n_neighbors": [3, 5, 7, 9, 11, 15, 21],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan", "minkowski"],
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=50,
    scoring="f1",
    cv=cv_strategy,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)

print("Starting RandomizedSearchCV (50 iterations, 5-fold CV)...")
print("Note: KNN on large datasets can be slow due to distance calculations.")
random_search.fit(X_train, y_train_enc)
print("\nSearch complete!")

Starting RandomizedSearchCV (50 iterations, 5-fold CV)...
Note: KNN on large datasets can be slow due to distance calculations.
Fitting 5 folds for each of 42 candidates, totalling 210 fits


C:\Users\ultim\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 42 is smaller than n_iter=50. Running 42 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [ ]:
print("Best hyperparameters found:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV F1-score: {random_search.best_score_:.4f}")

# Extract best model from the search
best_pipeline = random_search.best_estimator_

## 6. Probability Output & Custom Decision Threshold

KNN outputs probabilities based on the proportion of neighbors belonging to each class. We apply a **custom threshold** as an actionable business policy.

In [ ]:
# Predict probabilities on the test set
y_proba = best_pipeline.predict_proba(X_test)[:, 1]

print("Probability score summary (P(HeartDisease=Yes)):")
print(f"  Min:    {y_proba.min():.4f}")
print(f"  Max:    {y_proba.max():.4f}")
print(f"  Mean:   {y_proba.mean():.4f}")
print(f"  Median: {np.median(y_proba):.4f}")

# Apply default decision threshold
y_pred_default = (y_proba >= 0.5).astype(int)

print(f"\nPredictions with default threshold (0.50):")
print(f"  Predicted Yes: {np.sum(y_pred_default == 1)}, Predicted No: {np.sum(y_pred_default == 0)}")

In [ ]:
# Visualise the probability distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(y_proba[y_test_enc == 0], bins=50, alpha=0.6, label="Actual: No", color="steelblue")
ax.hist(y_proba[y_test_enc == 1], bins=50, alpha=0.6, label="Actual: Yes", color="salmon")
ax.axvline(0.5, color="red", linestyle="--", linewidth=2, label="Threshold = 0.5")
ax.set_xlabel("Predicted Probability of Heart Disease")
ax.set_ylabel("Count")
ax.set_title("Distribution of Predicted Probabilities (KNN)")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Threshold Optimization

Since KNN doesn't have built-in class weights, threshold adjustment is our primary method for handling class imbalance. We search for the optimal threshold that maximizes F1-score.

In [ ]:
# Sweep thresholds from 0.10 to 0.90 in steps of 0.05 (aligned with XGBoost notebook)
thresholds_sweep = np.arange(0.10, 0.91, 0.05)

results = []
for t in thresholds_sweep:
    y_pred_t = (y_proba >= t).astype(int)
    f1 = f1_score(y_test_enc, y_pred_t, zero_division=0)
    prec = precision_score(y_test_enc, y_pred_t, zero_division=0)
    rec = recall_score(y_test_enc, y_pred_t, zero_division=0)
    results.append({"Threshold": round(t, 2), "F1": f1, "Precision": prec, "Recall": rec})

threshold_df = pd.DataFrame(results)

print("Threshold Search Results")
print("=" * 60)
print(threshold_df.to_string(index=False))

# Find optimal threshold
best_row = threshold_df.loc[threshold_df["F1"].idxmax()]
OPTIMAL_THRESHOLD = best_row["Threshold"]
print(f"\nOptimal threshold (max F1): {OPTIMAL_THRESHOLD}")

In [ ]:
# Plot F1, Precision, Recall vs Threshold
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(threshold_df["Threshold"], threshold_df["F1"], "o-", color="green", linewidth=2, label="F1 Score")
ax.plot(threshold_df["Threshold"], threshold_df["Precision"], "s--", color="blue", linewidth=2, label="Precision")
ax.plot(threshold_df["Threshold"], threshold_df["Recall"], "^--", color="red", linewidth=2, label="Recall")
ax.axvline(OPTIMAL_THRESHOLD, color="grey", linestyle=":", linewidth=1.5, label=f"Optimal threshold ({OPTIMAL_THRESHOLD})")
ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Score")
ax.set_title("F1 / Precision / Recall vs. Decision Threshold (KNN)")
ax.legend()
ax.set_xticks(thresholds_sweep)
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Apply optimal threshold
y_pred_optimal = (y_proba >= OPTIMAL_THRESHOLD).astype(int)

print(f"Predictions with optimal threshold ({OPTIMAL_THRESHOLD}):")
print(f"  Predicted Yes: {np.sum(y_pred_optimal == 1)}, Predicted No: {np.sum(y_pred_optimal == 0)}")

## 8. Evaluation & Failure Analysis

### 8.1 Classification Report (Optimal Threshold)

In [ ]:
target_names = ["No Heart Disease (0)", "Heart Disease (1)"]
print(f"Classification Report (Threshold = {OPTIMAL_THRESHOLD})")
print("=" * 60)
print(classification_report(y_test_enc, y_pred_optimal, target_names=target_names))

### 8.2 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Default threshold
ConfusionMatrixDisplay.from_predictions(
    y_test_enc, y_pred_default,
    display_labels=target_names,
    cmap="Blues",
    ax=axes[0],
)
axes[0].set_title("Confusion Matrix (Default Threshold = 0.50)")

# Optimal threshold
ConfusionMatrixDisplay.from_predictions(
    y_test_enc, y_pred_optimal,
    display_labels=target_names,
    cmap="Blues",
    ax=axes[1],
)
axes[1].set_title(f"Confusion Matrix (Optimal Threshold = {OPTIMAL_THRESHOLD})")

plt.tight_layout()
plt.show()

### 8.3 ROC-AUC Score & Curve

In [ ]:
roc_auc = roc_auc_score(y_test_enc, y_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

fpr, tpr, thresholds_roc = roc_curve(y_test_enc, y_proba)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color="steelblue", linewidth=2, label=f"KNN (AUC = {roc_auc:.4f})")
ax.plot([0, 1], [0, 1], color="grey", linestyle="--", label="Random Classifier")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — KNN Classifier")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

### 8.4 Precision-Recall Curve

In [ ]:
precisions, recalls, pr_thresholds = precision_recall_curve(y_test_enc, y_proba)
pr_auc = auc(recalls, precisions)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recalls, precisions, color="steelblue", linewidth=2,
        label=f"KNN (PR-AUC = {pr_auc:.4f})")
ax.axhline(y=np.mean(y_test_enc), color="grey", linestyle="--",
           label=f"Baseline (prevalence = {np.mean(y_test_enc):.4f})")

# Mark the optimal threshold on the curve
idx_opt = np.argmin(np.abs(pr_thresholds - OPTIMAL_THRESHOLD))
ax.plot(recalls[idx_opt], precisions[idx_opt], "ro", markersize=10,
        label=f"Optimal threshold = {OPTIMAL_THRESHOLD} (P={precisions[idx_opt]:.2f}, R={recalls[idx_opt]:.2f})")

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve — KNN Classifier")
ax.legend(loc="upper right")
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.show()

print(f"PR-AUC Score: {pr_auc:.4f}")

### 8.5 Failure Analysis — Top 10 Most Confident Failures

These are cases where the model was **most confidently wrong**:
- **False Positives:** high predicted probability, but actual label is No (0)
- **False Negatives:** low predicted probability, but actual label is Yes (1)

Analysing these helps us understand the model's mechanical weaknesses.

In [ ]:
# Build a failure analysis dataframe
failure_df = pd.DataFrame({
    "actual": y_test_enc,
    "predicted_prob": y_proba,
    "predicted_label": y_pred_optimal,
})
failure_df["error"] = failure_df["actual"] != failure_df["predicted_label"]
failure_df["confidence"] = np.abs(failure_df["predicted_prob"] - 0.5)

# Filter to errors only, sorted by confidence (most confident mistakes first)
errors = failure_df[failure_df["error"]].sort_values("confidence", ascending=False)

print(f"Total misclassifications (threshold={OPTIMAL_THRESHOLD}): {len(errors)} / {len(failure_df)} ({len(errors)/len(failure_df)*100:.2f}%)")
print(f"\n{'='*80}")
print("TOP 10 MOST CONFIDENT FAILURES")
print(f"{'='*80}")

top_10 = errors.head(10).copy()
top_10["actual_label"] = le.inverse_transform(top_10["actual"])
top_10["failure_type"] = top_10.apply(
    lambda r: "FALSE POSITIVE (predicted Yes, actual No)" if r["actual"] == 0
    else "FALSE NEGATIVE (predicted No, actual Yes)",
    axis=1
)

for i, (idx, row) in enumerate(top_10.iterrows(), 1):
    print(f"\n#{i} | Test index: {idx}")
    print(f"   Predicted probability: {row['predicted_prob']:.4f}")
    print(f"   Actual label:          {row['actual_label']}")
    print(f"   Type:                  {row['failure_type']}")

In [ ]:
# Examine feature values of the top 10 failures
top_10_indices = top_10.index.tolist()
top_10_features = X_test.iloc[top_10_indices].copy()
top_10_features.insert(0, "actual_label", top_10["actual_label"].values)
top_10_features.insert(1, "predicted_prob", top_10["predicted_prob"].values)
top_10_features.insert(2, "failure_type", top_10["failure_type"].values)

print("Feature values for top 10 most confident failures:")
top_10_features.T

## 9. Feature Importance (Permutation Importance)

KNN does not have built-in feature importance like tree-based models (which use Mean Decrease in Impurity). Instead, we use **Permutation Importance** (as taught in Lab 05):

1. Compute baseline performance on test set
2. For each feature, randomly shuffle its values
3. Measure how much performance drops
4. Features causing larger drops are more important

This method is **model-agnostic** and provides reliable estimates of true predictive contribution, unlike MDI which can be biased toward high-cardinality features.

In [ ]:
# Compute permutation importance on test set
# Note: This may take some time due to KNN's computational nature
print("Computing permutation importance (this may take a few minutes)...")
perm_result = permutation_importance(
    best_pipeline,
    X_test,
    y_test_enc,
    n_repeats=10,
    random_state=RANDOM_STATE,
    scoring="f1",
    n_jobs=-1,
)

# Convert to DataFrame and sort
perm_importances = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance Mean": perm_result.importances_mean,
    "Importance Std": perm_result.importances_std
}).sort_values(by="Importance Mean", ascending=False)

print("\nPermutation Importance (Test Set) - Top 20 Features:")
print("=" * 60)
print(perm_importances.head(20).to_string(index=False))

In [ ]:
# Plot top 20 features
fig, ax = plt.subplots(figsize=(10, 8))
top_20 = perm_importances.head(20)
ax.barh(range(len(top_20)), top_20["Importance Mean"], xerr=top_20["Importance Std"], color="steelblue")
ax.set_yticks(range(len(top_20)))
ax.set_yticklabels(top_20["Feature"])
ax.set_xlabel("Mean Importance (F1 drop when shuffled)")
ax.set_title("Top 20 Feature Importances — KNN (Permutation Method)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 10. Comparison Metrics Summary

Summary table for comparison against groupmates' models.

In [ ]:
# Cross-validation metrics from the search
cv_results = pd.DataFrame(random_search.cv_results_)
best_idx = random_search.best_index_

mean_cv_f1 = cv_results.loc[best_idx, "mean_test_score"]
std_cv_f1 = cv_results.loc[best_idx, "std_test_score"]

# Test set metrics with optimal threshold
test_accuracy = accuracy_score(y_test_enc, y_pred_optimal)
test_f1_macro = f1_score(y_test_enc, y_pred_optimal, average="macro")
test_f1_yes = f1_score(y_test_enc, y_pred_optimal, average="binary")

summary = pd.DataFrame({
    "Model": ["KNN (tuned)"],
    "Algorithmic Family": ["Distance-Based"],
    "Best CV F1": [f"{mean_cv_f1:.4f} ± {std_cv_f1:.4f}"],
    "Test Accuracy": [f"{test_accuracy:.4f}"],
    "Test F1 (Macro)": [f"{test_f1_macro:.4f}"],
    "Test F1 (Yes)": [f"{test_f1_yes:.4f}"],
    "ROC-AUC": [f"{roc_auc:.4f}"],
    "PR-AUC": [f"{pr_auc:.4f}"],
    "Threshold": [OPTIMAL_THRESHOLD],
    "Best Params": [str(random_search.best_params_)],
})

print("=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)
summary.T

## 11. Pipeline Architecture Verification

This section demonstrates that our pipeline correctly prevents data leakage by encapsulating all transformations within the pipeline object.

In [ ]:
print("=" * 80)
print("PIPELINE ARCHITECTURE VERIFICATION")
print("=" * 80)
print("\nPipeline Structure:")
print(best_pipeline)

print("\n" + "-" * 80)
print("Data Leakage Prevention:")
print("-" * 80)
print("1. StandardScaler: Fitted ONLY on training data during each CV fold")
print("   - Mean and std computed from training fold only")
print("   - Test/validation fold transformed using training statistics")
print("\n2. KNeighborsClassifier: Trained ONLY on scaled training data")
print("   - Distance calculations use consistently scaled features")
print("\n3. Cross-Validation: StratifiedKFold ensures class balance in each fold")

print("\n" + "-" * 80)
print("Scaler Statistics (fitted on training data):")
print("-" * 80)
scaler = best_pipeline.named_steps["scaler"]
print(f"Number of features: {len(scaler.mean_)}")
print(f"\nFeature means (first 10): {scaler.mean_[:10].round(4)}")
print(f"Feature stds (first 10):  {scaler.scale_[:10].round(4)}")

## 12. Save Model Artifacts (Optional)

In [ ]:
import joblib

# Save the best pipeline
joblib.dump(best_pipeline, "knn_best_pipeline.joblib")
print("Best pipeline saved to: knn_best_pipeline.joblib")

# Save the full RandomizedSearchCV results
cv_results.to_csv("knn_cv_results.csv", index=False)
print("CV results saved to: knn_cv_results.csv")

# Save summary metrics
summary.to_csv("knn_summary_metrics.csv", index=False)
print("Summary metrics saved to: knn_summary_metrics.csv")